In [ ]:
import pandas as pd

roll = input("enter roll number: ")

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

own_entries = {
    "billing": {"question": "how do i get a refund", "answer": "Refunds are processed within 7 working days.",
                "keywords": "refund money return", "category": "billing"},
    "account": {"question": "how do i update my registered mobile number", "answer": "Go to Profile > Edit Details and verify with an OTP.",
                "keywords": "update mobile number", "category": "account"},
    "general": {"question": "where is the college located", "answer": "The college is on the main campus road.",
                "keywords": "college location address", "category": "general"},
}

categories = ["billing", "account", "general"]
entries = list(fixed_entries)
for ch in roll[-2:]:
    d = int(ch)
    cat = categories[d % 3]
    entry = dict(own_entries[cat])
    if entry in entries:
        entry["question"] = entry["question"] + " again"
    entries.append(entry)

df = pd.DataFrame(entries)
print(df)

In [ ]:
def score_query(query, df):
    words = query.lower().split()
    rows = []
    for i, row in df.iterrows():
        text = set(row["question"].lower().split() + row["keywords"].lower().split())
        hits = sum(1 for w in words if w in text)
        conf = hits / len(words)
        if conf > 0:
            rows.append((i, row["question"], row["answer"], conf))
    result = pd.DataFrame(rows, columns=["index", "question", "answer", "confidence"])
    return result.sort_values("confidence", ascending=False).reset_index(drop=True)

print(score_query("how to pay the fee", df))

In [ ]:
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"].tolist()

mycat = df.loc[4, "category"]
print(mycat)
print(same_category(mycat, df))

In [ ]:
new_kw = input("enter a new keyword: ")
df.loc[0, "keywords"] = df.loc[0, "keywords"] + " " + new_kw
print(df.loc[0])
df.to_csv(roll + "_faq_data.csv", index=False)

In [ ]:
print(df.groupby("category").size())

In [ ]:
def score_query(query, df):
    words = query.lower().split()
    rows = []
    for i, row in df.iterrows():
        text = set(row["question"].lower().split() + row["keywords"].lower().split())
        hits = sum(1 for w in words if w in text)
        conf = hits / len(words)
        if conf > 0:
            rows.append((i, row["question"], row["answer"], conf))
    result = pd.DataFrame(rows, columns=["index", "question", "answer", "confidence"])
    result = result.sort_values("confidence", ascending=False).reset_index(drop=True)
    if len(result) == 0:
        print("no match found")
        return result
    top = result[result["confidence"] == result["confidence"].max()]
    if len(top) > 1:
        print("tie between", len(top), "entries:")
        print(top)
    else:
        print("best match:")
        print(top)
    return result

r1 = score_query("fee", df)
r2 = score_query("reset password", df)